<a href="https://colab.research.google.com/github/geethat0126-bit/BIS_5TH_SEM/blob/main/week4_1BM24CS104.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import random

# Traffic Signal Optimization using Cuckoo Search

print("TRAFFIC SIGNAL TIMING OPTIMIZATION")
print("----------------------------------")

# User input
n = int(input("Enter number of traffic directions: "))

directions = []
arrival = []
queue = []
waiting = []

for i in range(n):

    print("\nDirection", i + 1)

    name = input("Enter direction name: ")
    directions.append(name)

    a = float(input("Enter vehicle arrival rate (vehicles/min): "))
    q = int(input("Enter queue length (vehicles): "))
    w = float(input("Enter average waiting time (minutes): "))

    arrival.append(a)
    queue.append(q)
    waiting.append(w)

# Signal cycle duration
cycle = float(input("\nEnter total signal cycle time (seconds): "))

# Cuckoo Search parameters
nests = 20
iterations = 100
pa = 0.25

min_green = 10
max_green = cycle - min_green * (n - 1)

if max_green < min_green * 0 + 10:
    raise ValueError("Cycle time is too small for the number of directions.")

# Fitness function
def fitness(solution):

    total = 0

    for i in range(n):

        # Estimated waiting time after signal allocation
        effective_wait = waiting[i] * (
            cycle / (solution[i] + 1)
        )

        # Congestion score
        total += (
            arrival[i] * effective_wait
            + queue[i] * effective_wait
        )

    return total


# Generate valid signal timings
def normalize(solution):

    solution = [
        max(min_green, min(x, cycle))
        for x in solution
    ]

    available = cycle - n * min_green

    extra = [max(0, x - min_green) for x in solution]
    total = sum(extra)

    if total == 0:
        return [cycle / n] * n

    return [
        min_green + (x / total) * available
        for x in extra
    ]


# Levy flight
def levy_flight(solution):

    new_solution = []

    for x in solution:
        step = random.gauss(0, 1)

        new_solution.append(x + step * 5)

    return normalize(new_solution)


# Initialize nests
population = []

for _ in range(nests):

    solution = [
        random.uniform(min_green, cycle / n)
        for _ in range(n)
    ]

    population.append(normalize(solution))


# Cuckoo Search
best_solution = None
best_fitness = float("inf")

for iteration in range(iterations):

    for i in range(nests):

        new_solution = levy_flight(population[i])

        new_fitness = fitness(new_solution)

        old_fitness = fitness(population[i])

        if new_fitness < old_fitness:
            population[i] = new_solution

    # Abandon worst nests
    population.sort(key=fitness)

    for i in range(int(pa * nests), nests):

        population[i] = normalize([
            random.uniform(min_green, cycle / n)
            for _ in range(n)
        ])

    # Update best solution
    current_best = min(population, key=fitness)

    if fitness(current_best) < best_fitness:
        best_fitness = fitness(current_best)
        best_solution = current_best[:]


# Display output
print("\n========== OPTIMIZED SIGNAL TIMINGS ==========")

for i in range(n):

    print(directions[i], ":", round(best_solution[i], 2),
          "seconds")

print("\nTotal Cycle Time:",
      round(sum(best_solution), 2), "seconds")

print("Minimum Congestion Fitness:",
      round(best_fitness, 2))

print("\nSignal timing optimization completed!")


TRAFFIC SIGNAL TIMING OPTIMIZATION
----------------------------------
Enter number of traffic directions: 4

Direction 1
Enter direction name: North
Enter vehicle arrival rate (vehicles/min): 20
Enter queue length (vehicles): 30
Enter average waiting time (minutes): 2

Direction 2
Enter direction name: South
Enter vehicle arrival rate (vehicles/min): 15
Enter queue length (vehicles): 20
Enter average waiting time (minutes): 3

Direction 3
Enter direction name: East
Enter vehicle arrival rate (vehicles/min): 25
Enter queue length (vehicles): 40
Enter average waiting time (minutes): 4

Direction 4
Enter direction name: West
Enter vehicle arrival rate (vehicles/min): 10
Enter queue length (vehicles): 15
Enter average waiting time (minutes): 2

Enter total signal cycle time (seconds): 120

========== OPTIMIZED SIGNAL TIMINGS ==========
North : 27.5 seconds
South : 28.52 seconds
East : 44.01 seconds
West : 19.98 seconds

Total Cycle Time: 120.0 seconds
Minimum Congestion Fitness: 1827.19

S